In [ ]:
pip install -q xgboost lightgbm catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 12.7 MB/s eta 0:00:00


**lfi  model comparison code **



In [3]:
# -*- coding: utf-8 -*-
"""
Modern Comparative Analysis for WAF - LFI Dataset
Author: Enhanced Version for Thesis
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# ====================== LOAD DATA csv======================
veriler = pd.read_csv("/content/drive/MyDrive/WAF_THESIS/DATASETS/lfi_cleaned.csv")   # Change path if needed

X = veriler.iloc[:, 0:7].values
y = veriler.iloc[:, 7:].values.ravel()   # Make 1D

print("Dataset Shape:", veriler.shape)
print("Class Distribution:\n", pd.Series(y).value_counts())

# ====================== PREPROCESSING ======================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ====================== HIGH-PERFORMANCE MODELS ======================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM (RBF)": SVC(kernel='rbf', C=10, probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                            random_state=42, eval_metric='logloss', use_label_encoder=False),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                              random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6,
                                  random_state=42, verbose=False)
}

# ====================== COMPARATIVE ANALYSIS ======================
results = []

print("\n" + "="*80)
print("COMPARATIVE ANALYSIS STARTED")
print("="*80)

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')

    results.append({
        'Model': name,
        'Accuracy': acc,
        'F1-Score (Macro)': f1,
        'Precision (Macro)': precision,
        'Recall (Macro)': recall
    })

    print(f"Accuracy : {acc:.5f}")
    print(f"F1-Score : {f1:.5f}")

# ====================== RESULTS TABLE ======================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='F1-Score (Macro)', ascending=False)

print("\n" + "="*80)
print("FINAL COMPARATIVE RESULTS")
print("="*80)
print(results_df.round(5))

# Save results
results_df.to_csv('model_comparison_results.csv', index=False)
print("\nResults saved to 'model_comparison_results.csv'")

# ====================== BEST MODEL ======================
best_model = results_df.iloc[0]
print(f"\n🏆 BEST MODEL: {best_model['Model']}")
print(f"Accuracy : {best_model['Accuracy']:.5f}")
print(f"F1-Score : {best_model['F1-Score (Macro)']:.5f}")

# Save best model
import joblib
best_model_name = best_model['Model'].replace(" ", "_").replace("(", "").replace(")", "")
joblib.dump(models[best_model['Model']], f'best_model_{best_model_name}.pkl')
print(f"Best model saved as: best_model_{best_model_name}.pkl")

Dataset Shape: (31177, 8)
Class Distribution:
 1    22716
0     8461
Name: count, dtype: int64

COMPARATIVE ANALYSIS STARTED

Training Logistic Regression...
Accuracy : 0.93794
F1-Score : 0.92443

Training SVM (RBF)...
Accuracy : 0.93826
F1-Score : 0.92479

Training Random Forest...
Accuracy : 0.93842
F1-Score : 0.92498

Training XGBoost...
Accuracy : 0.93842
F1-Score : 0.92498

Training LightGBM...
Accuracy : 0.93842
F1-Score : 0.92498

Training CatBoost...
Accuracy : 0.93842
F1-Score : 0.92498

FINAL COMPARATIVE RESULTS
                 Model  Accuracy  F1-Score (Macro)  Precision (Macro)  \
3              XGBoost   0.93842           0.92498            0.91182   
2        Random Forest   0.93842           0.92498            0.91182   
4             LightGBM   0.93842           0.92498            0.91182   
5             CatBoost   0.93842           0.92498            0.91182   
1            SVM (RBF)   0.93826           0.92479            0.91159   
0  Logistic Regression   0.93794  

**XSS model comparison code**

In [ ]:
# ====================== XSS DATASET - FIXED COMPARATIVE ANALYSIS ======================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

import joblib

print("🚀 Starting XSS Dataset Analysis (Fixed Version)...\n")

# ====================== LOAD DATA ======================
veriler = pd.read_csv("/content/drive/MyDrive/WAF_THESIS/DATASETS/xss_cleaned.csv")

X = veriler.iloc[:, 0:6]      # Features
y = veriler.iloc[:, 6:].values.ravel()   # Target

print("Dataset Shape:", veriler.shape)
print("Class Distribution:\n", pd.Series(y).value_counts())

# ====================== HANDLE MISSING VALUES ======================
print(f"NaN values before cleaning: {X.isna().sum().sum()}")

# Fill NaN with 0 (best for count-based features)
X = X.fillna(0)

# ====================== PREPROCESSING ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ====================== MODELS ======================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM (RBF)": SVC(kernel='rbf', C=10, probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                            random_state=42, eval_metric='logloss', use_label_encoder=False),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                              random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6,
                                  random_state=42, verbose=False)
}

# ====================== TRAINING & EVALUATION ======================
results = []

print("\n" + "="*85)
print("COMPARATIVE ANALYSIS - XSS DATASET (Fixed)")
print("="*85)

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')

    results.append({
        'Model': name,
        'Accuracy': acc,
        'F1-Score (Macro)': f1,
        'Precision (Macro)': precision,
        'Recall (Macro)': recall
    })

    print(
        f"✅ {name:20} → "
        f"Accuracy: {acc:.5f} | "
        f"F1: {f1:.5f} | "
        f"Precision: {precision:.5f} | "
        f"Recall: {recall:.5f}"
    )

# ====================== RESULTS ======================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by=['F1-Score (Macro)', 'Accuracy'],
    ascending=False
).reset_index(drop=True)

print("\n" + "="*100)
print("🏆 FINAL RESULTS - XSS DATASET")
print("="*100)
print(results_df.round(5))

results_df.to_csv('xss_model_comparison_fixed.csv', index=False)
print("\n📊 Results saved to 'xss_model_comparison_fixed.csv'")

# ====================== BEST MODEL ======================
best = results_df.iloc[0]

print(f"\n🎯 BEST MODEL: {best['Model']}")
print(f"   Accuracy  : {best['Accuracy']:.5f}")
print(f"   F1-Score  : {best['F1-Score (Macro)']:.5f}")
print(f"   Precision : {best['Precision (Macro)']:.5f}")
print(f"   Recall    : {best['Recall (Macro)']:.5f}")

best_name = best['Model'].replace(" ", "_").replace("(", "").replace(")", "")
joblib.dump(models[best['Model']], f'best_xss_model_{best_name}.pkl')
print(f"💾 Best model saved as 'best_xss_model_{best_name}.pkl'!")

🚀 Starting XSS Dataset Analysis (Fixed Version)...

Dataset Shape: (30461, 7)
Class Distribution:
 1    19770
0    10691
Name: count, dtype: int64
NaN values before cleaning: 0

COMPARATIVE ANALYSIS - XSS DATASET (Fixed)

Training Logistic Regression...
✅ Logistic Regression  → Accuracy: 0.99721 | F1: 0.99694 | Precision: 0.99647 | Recall: 0.99742

Training SVM (RBF)...
✅ SVM (RBF)            → Accuracy: 0.99803 | F1: 0.99784 | Precision: 0.99721 | Recall: 0.99848

Training Random Forest...
✅ Random Forest        → Accuracy: 0.99754 | F1: 0.99730 | Precision: 0.99683 | Recall: 0.99778

Training XGBoost...
✅ XGBoost              → Accuracy: 0.99803 | F1: 0.99784 | Precision: 0.99721 | Recall: 0.99848

Training LightGBM...
✅ LightGBM             → Accuracy: 0.99803 | F1: 0.99784 | Precision: 0.99721 | Recall: 0.99848

Training CatBoost...
✅ CatBoost             → Accuracy: 0.99803 | F1: 0.99784 | Precision: 0.99721 | Recall: 0.99848

🏆 FINAL RESULTS - XSS DATASET
                 Model  

**OSC model comparison code**

In [ ]:
# ====================== OSC DATASET - COMPARATIVE ANALYSIS ======================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

import joblib

print("🚀 Starting OSC Dataset Analysis...\n")

# ====================== LOAD OSC DATASET ======================
veriler = pd.read_csv("/content/drive/MyDrive/WAF_THESIS/DATASETS/osc_cleaned.csv")

# OSC has 8 features + 1 target
X = veriler.iloc[:, 0:8]      # First 8 columns are features
y = veriler.iloc[:, 8:].values.ravel()   # Target column

print("Dataset Shape:", veriler.shape)
print("Columns:", veriler.columns.tolist())
print("\nClass Distribution:\n", pd.Series(y).value_counts())

# ====================== HANDLE MISSING VALUES ======================
print(f"NaN values before cleaning: {X.isna().sum().sum()}")

# Fill NaN values with 0
X = X.fillna(0)

# ====================== PREPROCESSING ======================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ====================== MODELS ======================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM (RBF)": SVC(kernel='rbf', C=10, probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        verbose=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        random_state=42,
        verbose=False
    )
}

# ====================== TRAINING & EVALUATION ======================
results = []

print("\n" + "="*100)
print("COMPARATIVE ANALYSIS - OSC DATASET")
print("="*100)

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')

    results.append({
        'Model': name,
        'Accuracy': acc,
        'F1-Score (Macro)': f1,
        'Precision (Macro)': precision,
        'Recall (Macro)': recall
    })

    print(
        f"✅ {name:20} → "
        f"Accuracy: {acc:.5f} | "
        f"F1: {f1:.5f} | "
        f"Precision: {precision:.5f} | "
        f"Recall: {recall:.5f}"
    )

# ====================== FINAL RESULTS ======================
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=['F1-Score (Macro)', 'Accuracy'],
    ascending=False
).reset_index(drop=True)

print("\n" + "="*100)
print("🏆 FINAL RESULTS - OSC DATASET")
print("="*100)

print(results_df.round(5))

# Save results
results_df.to_csv('osc_model_comparison.csv', index=False)
print("\n📊 Results saved to 'osc_model_comparison.csv'")

# ====================== BEST MODEL ======================
best = results_df.iloc[0]

print(f"\n🎯 BEST MODEL FOR OSC: {best['Model']}")
print(f"   Accuracy  : {best['Accuracy']:.5f}")
print(f"   F1-Score  : {best['F1-Score (Macro)']:.5f}")
print(f"   Precision : {best['Precision (Macro)']:.5f}")
print(f"   Recall    : {best['Recall (Macro)']:.5f}")

# ====================== SAVE BEST MODEL ======================
best_name = best['Model'].replace(" ", "_").replace("(", "").replace(")", "")

joblib.dump(
    models[best['Model']],
    f'best_osc_model_{best_name}.pkl'
)

print(f"💾 Best model saved as: best_osc_model_{best_name}.pkl")

🚀 Starting OSC Dataset Analysis...

Dataset Shape: (19095, 9)
Columns: ['dashes', 'and_sign', 'dolar_sign', 'or_sign', 'lessthan', 'greaterthan', 'exclamation', 'badwords', 'class']

Class Distribution:
 1    10679
0     8416
Name: count, dtype: int64
NaN values before cleaning: 0

COMPARATIVE ANALYSIS - OSC DATASET

Training Logistic Regression...
✅ Logistic Regression  → Accuracy: 0.98219 | F1: 0.98196 | Precision: 0.98156 | Recall: 0.98238

Training SVM (RBF)...
✅ SVM (RBF)            → Accuracy: 0.98612 | F1: 0.98596 | Precision: 0.98480 | Recall: 0.98747

Training Random Forest...
✅ Random Forest        → Accuracy: 0.98638 | F1: 0.98623 | Precision: 0.98505 | Recall: 0.98776

Training XGBoost...
✅ XGBoost              → Accuracy: 0.98638 | F1: 0.98623 | Precision: 0.98505 | Recall: 0.98776

Training LightGBM...
✅ LightGBM             → Accuracy: 0.98638 | F1: 0.98623 | Precision: 0.98505 | Recall: 0.98776

Training CatBoost...
✅ CatBoost             → Accuracy: 0.98638 | F1: 0.986

SQL model comparison

In [ ]:
# ====================== SQL DATASET - COMPARATIVE ANALYSIS ======================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

import joblib

print("🚀 Starting SQL Dataset Analysis...\n")

# ====================== LOAD SQL DATASET ======================
veriler = pd.read_csv("/content/drive/MyDrive/WAF_THESIS/DATASETS/sql_cleaned.csv")

# SQL dataset has 8 features + target
X = veriler.iloc[:, 0:8]      # Features
y = veriler.iloc[:, 8:].values.ravel()   # Target column

print("Dataset Shape:", veriler.shape)
print("Columns:", veriler.columns.tolist())
print("\nClass Distribution:\n", pd.Series(y).value_counts())

# ====================== HANDLE MISSING VALUES ======================
print(f"NaN values before cleaning: {X.isna().sum().sum()}")

# Fill NaN values with 0
X = X.fillna(0)

# ====================== PREPROCESSING ======================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ====================== MODELS ======================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),

    "SVM (RBF)": SVC(
        kernel='rbf',
        C=10,
        probability=True,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        verbose=-1
    ),

    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        random_state=42,
        verbose=False
    )
}

# ====================== TRAINING & EVALUATION ======================
results = []

print("\n" + "="*100)
print("COMPARATIVE ANALYSIS - SQL DATASET")
print("="*100)

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')

    results.append({
        'Model': name,
        'Accuracy': acc,
        'F1-Score (Macro)': f1,
        'Precision (Macro)': precision,
        'Recall (Macro)': recall
    })

    print(
        f"✅ {name:20} → "
        f"Accuracy: {acc:.5f} | "
        f"F1: {f1:.5f} | "
        f"Precision: {precision:.5f} | "
        f"Recall: {recall:.5f}"
    )

# ====================== FINAL RESULTS ======================
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=['F1-Score (Macro)', 'Accuracy'],
    ascending=False
).reset_index(drop=True)

print("\n" + "="*100)
print("🏆 FINAL RESULTS - SQL DATASET")
print("="*100)

print(results_df.round(5))

# Save results
results_df.to_csv('sql_model_comparison.csv', index=False)
print("\n📊 Results saved to 'sql_model_comparison.csv'")

# ====================== BEST MODEL ======================
best = results_df.iloc[0]

print(f"\n🎯 BEST MODEL FOR SQL: {best['Model']}")
print(f"   Accuracy  : {best['Accuracy']:.5f}")
print(f"   F1-Score  : {best['F1-Score (Macro)']:.5f}")
print(f"   Precision : {best['Precision (Macro)']:.5f}")
print(f"   Recall    : {best['Recall (Macro)']:.5f}")

# ====================== SAVE BEST MODEL ======================
best_name = best['Model'].replace(" ", "_").replace("(", "").replace(")", "")

joblib.dump(
    models[best['Model']],
    f'best_sql_model_{best_name}.pkl'
)

print(f"💾 Best model saved as: best_sql_model_{best_name}.pkl")

🚀 Starting SQL Dataset Analysis...

Dataset Shape: (13477, 9)
Columns: ['stars', 'dashes', 'or_sign', 'and_sign', 'sub_line', 'comment_sign', 'at_sign', 'badwords', 'class']

Class Distribution:
 0    10190
1     3287
Name: count, dtype: int64
NaN values before cleaning: 0

COMPARATIVE ANALYSIS - SQL DATASET

Training Logistic Regression...
✅ Logistic Regression  → Accuracy: 0.84570 | F1: 0.72646 | Precision: 0.90082 | Recall: 0.68801

Training SVM (RBF)...
✅ SVM (RBF)            → Accuracy: 0.99518 | F1: 0.99350 | Precision: 0.99078 | Recall: 0.99630

Training Random Forest...
✅ Random Forest        → Accuracy: 0.99481 | F1: 0.99301 | Precision: 0.99005 | Recall: 0.99605

Training XGBoost...
✅ XGBoost              → Accuracy: 0.99481 | F1: 0.99301 | Precision: 0.99005 | Recall: 0.99605

Training LightGBM...
✅ LightGBM             → Accuracy: 0.99369 | F1: 0.99149 | Precision: 0.98975 | Recall: 0.99326

Training CatBoost...
✅ CatBoost             → Accuracy: 0.99518 | F1: 0.99350 | Pre

**Final Decision for best model**

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# ====================== DEFINE BEST MODELS PER DATASET ======================

best_models = {
    "LFI": "XGBoost",      # Change according to your actual results
    "XSS": "XGBoost",
    "OSC": "XGBoost",
    "SQL": "SVM (RBF)"     # You found SVM best for SQL
}

print("🎯 Selected Best Models per Dataset:")
for ds, model in best_models.items():
    print(f"   {ds:6} → {model}")

# ====================== LOAD AND EVALUATE BEST MODELS ======================

datasets_info = {
    "LFI": "/content/drive/MyDrive/preprocessed_data/lfi_cleaned.csv",
    "XSS": "/content/drive/MyDrive/preprocessed_data/xss_cleaned.csv",
    "OSC": "/content/drive/MyDrive/preprocessed_data/osc_cleaned.csv",
    "SQL": "/content/drive/MyDrive/preprocessed_data/sql_cleaned.csv"
}

final_results = []

for ds_name, file_path in datasets_info.items():
    print(f"\n{'='*60}")
    print(f"Evaluating Best Model for {ds_name}")
    print(f"{'='*60}")

    df = pd.read_csv(file_path)
    X = df.iloc[:, :-1].fillna(0)
    y = df.iloc[:, -1]

    # Convert target to 0/1 if needed
    if y.dtype == 'object' or y.dtype == 'str':
        y = y.astype(str).str.lower().map({'good':0, 'bad':1, '0':0, '1':1})

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model_name = best_models[ds_name]

    # Load or train best model
    if model_name == "SVM (RBF)":
        model = SVC(kernel='rbf', C=10, probability=True, random_state=42)
    elif model_name == "XGBoost":
        model = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=7, random_state=42)
    else:
        model = XGBClassifier(n_estimators=300, random_state=42)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    final_results.append({
        'Dataset': ds_name,
        'Best_Model': model_name,
        'Accuracy': acc,
        'F1_Score': f1
    })

    # Save best model
    joblib.dump(model, f'final_best_model_{ds_name}.pkl')
    print(f"✅ {ds_name} → {model_name} | Accuracy: {acc:.5f} | F1: {f1:.5f}")

# Final Summary
final_df = pd.DataFrame(final_results)
print("\n" + "="*70)
print("FINAL BEST MODEL SUMMARY")
print("="*70)
print(final_df.round(5))

final_df.to_csv('final_best_models_summary.csv', index=False)

🎯 Selected Best Models per Dataset:
   LFI    → XGBoost
   XSS    → XGBoost
   OSC    → XGBoost
   SQL    → SVM (RBF)

Evaluating Best Model for LFI
✅ LFI → XGBoost | Accuracy: 0.93842 | F1: 0.92498

Evaluating Best Model for XSS
✅ XSS → XGBoost | Accuracy: 0.99787 | F1: 0.99766

Evaluating Best Model for OSC
✅ OSC → XGBoost | Accuracy: 0.98638 | F1: 0.98623

Evaluating Best Model for SQL
✅ SQL → SVM (RBF) | Accuracy: 0.99518 | F1: 0.99350

FINAL BEST MODEL SUMMARY
  Dataset Best_Model  Accuracy  F1_Score
0     LFI    XGBoost   0.93842   0.92498
1     XSS    XGBoost   0.99787   0.99766
2     OSC    XGBoost   0.98638   0.98623
3     SQL  SVM (RBF)   0.99518   0.99350
